# State 와 reducer

강의 오후 ② 가운데 (Academy 모듈 2 · L1 State Schema · L2 Reducers · L3 Multiple Schemas)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① State | schema 로 필드를 적고 돌려 본다. 노드를 지날 때마다 필드가 채워진다. 타입이 틀린 값은 Annotation 이면 그냥 통과, zod 면 에러 |
| ② reducer | 기본은 덮어쓰기. 같은 단계에서 두 노드가 같은 필드에 쓰면 에러 → 이어 붙이기 · 직접 만든 규칙 |
| ③ messages reducer | 뒤에 붙이기 · 같은 id 면 덮어쓰기 · RemoveMessage 로 지우기 |
| ④ 여러 schema | 노드마다 받는 필드를 정한다 (private state). 그래프의 입구 · 출구를 따로 정한다. State 에 없는 필드는 버려진다 |

이 노트북은 LLM 을 부르지 않습니다. State 가 어떻게 채워지고 합쳐지는지에만 집중하겠습니다.

In [ ]:
import "npm:@langchain/core@1.2.14";
import { Annotation, MessagesAnnotation, messagesStateReducer, StateGraph, START, END } from "npm:@langchain/langgraph@1.4.18";
import { HumanMessage, AIMessage, RemoveMessage } from "npm:@langchain/core@1.2.14/messages";
import { z } from "npm:zod@4.6.7";

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## ① State 와 schema

State 는 그래프가 도는 동안 노드들이 함께 들고 다니는 객체 하나입니다. 이 객체에 어떤 필드가 있고 필드마다 무엇이 들어가는지 적어 둔 것을 **schema** 라고 부릅니다. `Annotation.Root` 로 적습니다.

오후 ① 의 이메일 분석 그래프를 다시 만들어 보겠습니다. 필드는 셋입니다. 학생이 쓴 글 `text`, 글의 종류 `kind`, 그리고 `feedback`. 노드마다 받은 State 를 찍어 보겠습니다.

In [ ]:
// L1 · schema: State 에 어떤 필드가 있고, 필드마다 무엇이 들어가는지
const EmailState = Annotation.Root({
  text: Annotation<string>(),
  kind: Annotation<"email" | "ielts">(),
  feedback: Annotation<string>(),
});
const email = new StateGraph(EmailState)
  .addNode("classify", (s) => { console.log("[NODE] ▶ classify      받은 State", s); return { kind: "email" as const }; })
  .addNode("analyze_email", (s) => { console.log("[NODE] ▶ analyze_email 받은 State", s); return { feedback: "맺음말이 빠졌다" }; })
  .addEdge(START, "classify").addEdge("classify", "analyze_email").addEdge("analyze_email", END)
  .compile();
await showGraph(email);
console.log("[끝]   마지막 State", await email.invoke({ text: "Dear Mr. Kim, I am writing to ask about the schedule..." }));

**돌려 보면** classify 는 `text` 만 있는 State 를 받고, analyze_email 은 `kind` 까지 채워진 State 를 받습니다. 마지막 State 에는 `feedback` 까지 세 필드가 다 있습니다.

노드는 State 전체를 다시 만들지 않습니다. classify 는 `kind` 만, analyze_email 은 `feedback` 만 돌려줬습니다. 나머지 필드는 그대로 넘어갔습니다.

### 타입이 틀리면?

schema 에 kind 는 "email" 아니면 "ielts" 라고 적었습니다. 코드에 `kind: "poem"` 이라고 쓰면 편집기와 tsc 가 빨간 줄을 긋습니다. 그런데 실행 중에 밖에서 들어온 값, 예를 들어 사용자 입력이나 모델의 답이라면 어떨까요? `as any` 로 타입 검사를 건너뛰어, 밖에서 들어온 값처럼 넣어 보겠습니다.

In [ ]:
// 타입이 틀리면? — 실행 중에는 검사하지 않는다. 밖에서 들어온 값처럼 any 로 넣어 보면 그냥 통과한다
await email.invoke({ text: "Dear Mr. Kim, ...", kind: "poem" } as any);

**돌려 보면** 에러 없이 classify 와 analyze_email 이 돕니다. classify 는 kind = "poem" 을 받습니다. TypeScript 타입은 코드를 쓸 때만 검사하고, 실행 중에는 아무도 확인하지 않습니다. 그래서 밖에서 들어온 값은 노드에서 한 번 확인하고 쓰는 게 안전합니다.

### 실행 중에도 막으려면 — zod

**zod 는 무엇인가.** TypeScript 에서 많이 쓰는 데이터 검사 라이브러리입니다. "이 값은 글이어야 하고, 이 값은 email 아니면 ielts 중 하나여야 한다" 같은 규칙을 코드로 적어 두면, 실행 중에 들어온 값이 그 규칙에 맞는지 확인해 줍니다. 맞지 않으면 어느 필드가 왜 틀렸는지 알려 주는 에러를 냅니다.

- TypeScript 타입은 실행할 때 사라지지만, zod 의 규칙은 실행 중에도 남아서 값을 검사합니다
- 규칙에서 TypeScript 타입도 뽑아 줍니다 (`z.infer`). 타입을 두 번 적지 않아도 됩니다
- API 로 받은 JSON, 사용자 입력, 모델의 답처럼 밖에서 들어오는 값을 검사할 때 주로 씁니다. 도구의 입력 모양을 적을 때도 많이 씁니다
- Python 의 Pydantic 과 비슷한 자리입니다. Academy 강의는 이 장면을 Pydantic 으로 보여 줍니다

LangGraph 는 zod 로 적은 schema 를 State 로 그대로 받습니다. schema 를 Annotation 대신 **zod** 로 적어 보겠습니다. `z.object` 로 필드를 적고, 그대로 `new StateGraph(...)` 에 넘깁니다. 노드와 엣지는 위와 똑같습니다.

이번에는 classify 가 모델처럼 답한다고 치겠습니다. 이메일이면 "email", 아니면 "poem" 을 돌려줍니다. 정해 둔 값이 아닌 "poem" 이 들어오면 어떻게 될까요?

In [ ]:
// zod 로 schema 를 적으면 실행 중에도 검사한다. 노드와 엣지는 위와 똑같다
const EmailZod = z.object({
  text: z.string(),
  kind: z.enum(["email", "ielts"]).optional(),
  feedback: z.string().optional(),
});
const emailZod = new StateGraph(EmailZod)
  .addNode("classify", (s) => {
    const modelSays = s.text.startsWith("Dear") ? "email" : "poem";   // 모델이 이렇게 답했다고 치자
    console.log("[NODE] ▶ classify      →", modelSays);
    return { kind: modelSays as any };
  })
  .addNode("analyze_email", (s) => { console.log("[NODE] ▶ analyze_email 받은 State", s); return { feedback: "맺음말이 빠졌다" }; })
  .addEdge(START, "classify").addEdge("classify", "analyze_email").addEdge("analyze_email", END)
  .compile();
await showGraph(emailZod);
console.log("[끝]   마지막 State", await emailZod.invoke({ text: "Dear Mr. Kim, ..." }));
try {
  await emailZod.invoke({ text: "Roses are red, violets are blue ..." });
} catch (e) {
  console.log("[ERROR]", (e as Error).name, JSON.parse((e as Error).message)[0].message);
}

**돌려 보면** 이메일은 앞과 똑같이 끝까지 돕니다. 시를 넣으면 classify 가 "poem" 을 돌려주는 순간 `ZodError` 가 납니다. `Invalid option: expected one of "email"|"ielts"`. analyze_email 은 돌지 않습니다.

zod 는 값이 State 에 들어오는 순간 검사합니다. `as any` 로 TypeScript 를 속여도 잡힙니다. invoke 에 넣는 입력도 같은 방법으로 검사합니다.

## ② reducer

reducer 는 노드가 돌려준 값을 **State 에 어떻게 합칠지** 정하는 규칙입니다. 필드마다 따로 정합니다. 먼저 reducer 를 안 적었을 때입니다.

In [ ]:
// L2 · reducer 를 안 적으면 덮어쓴다
const Draft = Annotation.Root({ revision: Annotation<number>() });
const revise = new StateGraph(Draft)
  .addNode("revise", (s) => ({ revision: s.revision + 1 }))
  .addEdge(START, "revise").addEdge("revise", END)
  .compile();
await showGraph(revise);
console.log(await revise.invoke({ revision: 1 }));

**돌려 보면** `{ revision: 2 }` 입니다. LangGraph 는 이 필드를 어떻게 합치고 싶은지 모릅니다. 그래서 기본은 **덮어쓰기** 입니다.

이번에는 문법 분석과 어휘 분석을 **같은 단계** 에서 돌려 보겠습니다. LangGraph 는 그래프를 단계별로 돕니다. 한 단계에서 실행할 노드들을 함께 돌리고, 단계가 끝나면 돌려받은 값을 한꺼번에 State 에 합칩니다. START 에서 두 노드로 선을 하나씩 그으면 둘은 같은 단계에 들어갑니다. 그리고 둘 다 `notes` 필드에 씁니다.

In [ ]:
// 같은 단계에서 두 노드가 같은 필드에 쓰면? — 문법 분석과 어휘 분석을 동시에 돌린다
const Notes = Annotation.Root({ notes: Annotation<string[]>() });
const twoAnalyzers = (S: any) => new StateGraph(S)
  .addNode("grammar", () => ({ notes: ["문법: 3인칭 단수 -s 가 빠졌다"] }))
  .addNode("vocab", () => ({ notes: ["어휘: ubiquitous 는 너무 어렵다"] }))
  .addEdge(START, "grammar").addEdge(START, "vocab")     // START 에서 둘 다 출발 = 같은 단계
  .addEdge("grammar", END).addEdge("vocab", END)
  .compile();
await showGraph(twoAnalyzers(Notes));
try {
  await twoAnalyzers(Notes).invoke({});
} catch (e) {
  console.log("[ERROR]", (e as Error).name, "\n", (e as Error).message.split("\n")[0]);
}

**돌려 보면** `InvalidUpdateError` 가 납니다. `LastValue can only receive one value per step`, 덮어쓰는 필드는 한 단계에 값을 하나만 받을 수 있다는 뜻입니다. 한 단계에 값이 둘 왔으니 어느 쪽을 남길지 정할 수가 없습니다.

`notes` 필드에 "이어 붙인다" 는 reducer 를 달아 보겠습니다.

In [ ]:
// 필드에 이어 붙이는 reducer 를 달면, 둘 다 남는다
const NotesWithReducer = Annotation.Root({
  notes: Annotation<string[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
console.log(await twoAnalyzers(NotesWithReducer).invoke({}));

**돌려 보면** 에러 없이 두 줄이 다 남습니다.

reducer 는 함수 하나라서, 규칙을 **직접** 만들 수도 있습니다. 단어장 필드에 두 분석기가 모두 `mitigate` 를 넣었습니다. 이어 붙이되 겹치는 건 빼는 규칙을 만들어 보겠습니다.

In [ ]:
// 직접 만든 reducer: 단어장. 두 분석기가 같은 단어를 넣어도 한 번만 남긴다
const Words = Annotation.Root({
  words: Annotation<string[]>({
    reducer: (prev, next) => [...new Set([...prev, ...next])],   // 이어 붙이고, 겹치는 건 뺀다
    default: () => [],
  }),
});
const wordbook = new StateGraph(Words)
  .addNode("grammar", () => ({ words: ["ubiquitous", "mitigate"] }))
  .addNode("vocab", () => ({ words: ["mitigate", "meticulous"] }))
  .addEdge(START, "grammar").addEdge(START, "vocab")
  .addEdge("grammar", END).addEdge("vocab", END)
  .compile();
await showGraph(wordbook);
console.log(await wordbook.invoke({}));

**돌려 보면** `ubiquitous`, `mitigate`, `meticulous` 세 개만 남습니다.

## ③ messages 전용 reducer

대화(messages)는 거의 모든 Agent 에 있습니다. 그래서 LangGraph 에는 messages 를 위한 reducer 가 미리 들어 있습니다. `messagesStateReducer` 이고, `MessagesAnnotation` 을 State 로 쓰면 이게 달린 `messages` 필드가 생깁니다.

### 메시지는 왜 객체로 다루나

오전과 오후 ① 에서는 메시지를 `{ role: "user", content: "…" }` 로 적었습니다. 모델 API 가 받는 모양이 이것이기 때문입니다. OpenAI 가 정한 모양인데 사실상 표준이 돼서, 우리가 쓰는 OpenRouter 도 이 모양을 받습니다.

그런데 LangGraph 의 messages 는 이 모양이 아니라 `HumanMessage` · `AIMessage` 같은 **메시지 객체**로 다룹니다. 이유는 둘입니다.

1. **메시지를 가리킬 이름(id)이 필요하다.** `{ role, content }` 에는 id 가 없습니다. 그래서 "2번 메시지를 고쳐라", "1번 메시지를 지워라" 라고 말할 방법이 없습니다. 메시지 객체에는 id 가 붙어서, 아래에서 볼 덮어쓰기와 지우기를 할 수 있습니다.
2. **모델 회사마다 받는 모양이 조금씩 다르다.** Anthropic 은 system 을 따로 받고, Gemini 는 content 대신 parts 를 씁니다. LangChain 은 그래프 안에서는 한 가지 모양(메시지 객체)으로 다루고, 모델을 부를 때 회사에 맞게 바꿔 보냅니다.

user 는 `HumanMessage`, assistant 는 `AIMessage` 가 됩니다. 아래 셀에서는 id 를 직접 정해 주려고 메시지 객체로 만들겠습니다.

이 reducer 는 세 가지를 합니다. 먼저 뒤에 붙이기.

In [ ]:
// messages 전용 reducer: MessagesAnnotation 에 들어 있는 messagesStateReducer
const show = (ms: any[]) => ms.map((m) => `${m.id} · ${m.getType()} · ${m.content}`);
const talk = [
  new HumanMessage({ content: "ubiquitous 뜻이 뭐야?", id: "1" }),
  new AIMessage({ content: "어디에나 있는", id: "2" }),
];
console.log(show(messagesStateReducer(talk, [new HumanMessage({ content: "mitigate 는?", id: "3" })])));   // ① 뒤에 붙인다

**돌려 보면** 3번 메시지가 뒤에 붙었습니다. 여기까지는 우리가 만든 이어 붙이기와 같습니다.

다른 점은 메시지마다 **id** 가 있다는 것입니다. 같은 id 의 메시지를 돌려주면, 붙이지 않고 그 메시지를 덮어씁니다.

In [ ]:
// ② 같은 id 면 덮어쓴다
console.log(show(messagesStateReducer(talk, [new AIMessage({ content: "어디에나 있는 (= everywhere)", id: "2" })])));

**돌려 보면** 2번 메시지의 내용만 바뀌었습니다. 메시지는 여전히 두 개입니다.

그리고 `RemoveMessage` 에 id 를 담아 돌려주면, 그 메시지를 **지웁니다.**

In [ ]:
// ③ RemoveMessage 로 id 를 지정하면 지운다
console.log(show(messagesStateReducer(talk, [new RemoveMessage({ id: "1" })])));

**돌려 보면** 1번이 지워지고 2번만 남았습니다. 이 "지우기" 는 뒤에서 긴 대화를 정리할 때 씁니다.

그래프에서는 `MessagesAnnotation` 을 State 로 넘기면 됩니다. 이때 메시지를 꼭 객체로 만들 필요는 없습니다. 오전부터 써 온 `{ role, content }` 모양으로 넣어도, reducer 가 메시지 객체로 바꾸고 id 도 붙여 줍니다. 입력도, 노드가 돌려주는 값도 평범한 모양으로 적어 보겠습니다.

In [ ]:
// 그래프에서는 MessagesAnnotation 을 State 로 쓰면 된다. { role, content } 로 넣어도 메시지 객체로 바꿔 준다
const chatGraph = new StateGraph(MessagesAnnotation)
  .addNode("tutor", () => ({ messages: [{ role: "assistant", content: "어디에나 있는" }] }))
  .addEdge(START, "tutor").addEdge("tutor", END)
  .compile();
await showGraph(chatGraph);
const out = await chatGraph.invoke({ messages: [{ role: "user", content: "ubiquitous 뜻이 뭐야?" }] });
console.log(out.messages.map((m) => `${m.getType()} · ${m.content} · id ${m.id?.slice(0, 8)}…`));

**돌려 보면** human, ai 두 메시지가 객체로 바뀌어 있고, 각각 id 가 붙었습니다.

오후 ① 과 실습에서는 messages 를 `{ role, content }` 객체로 직접 다루고, 이어 붙이는 reducer 를 직접 달았습니다. 같은 일을 하는 부품이 미리 만들어져 있다는 것만 알아 두면 됩니다.

## ④ 여러 schema

지금까지는 모든 노드가 같은 State 하나를 받았습니다. schema 를 여러 개 두면 노드마다, 또 그래프의 입구와 출구마다 보는 필드를 정할 수 있습니다. Academy 와 LangGraph 문서는 이걸 **multiple schemas** 라고 부릅니다.

### 노드마다 받는 State — private state

analyze 가 분석 메모 `notes` 를 남기고, write 가 그 메모로 피드백을 씁니다. 그런데 notes 는 write 만 쓰면 되고, 그래프 밖으로 나갈 필요는 없습니다.

그래서 notes 는 그래프 State(`Essay`) 에 넣지 않고 따로 `Private` schema 에 적습니다. 그리고 write 노드를 등록할 때 `{ input: Private }` 를 줍니다. 이 노드는 Private 의 필드만 받는다는 뜻입니다.

In [ ]:
// L3 · private state: write 노드는 Private 의 필드(notes)만 받는다. notes 는 그래프 State 에 없다
const Essay = Annotation.Root({ text: Annotation<string>(), feedback: Annotation<string>() });
const Private = Annotation.Root({ notes: Annotation<string[]>() });
const privateDemo = new StateGraph(Essay)
  .addNode("analyze", (s) => { console.log("[NODE] ▶ analyze 받은 State", s); return { notes: ["3인칭 단수 -s"] }; })
  .addNode("write", (s) => { console.log("[NODE] ▶ write   받은 State", s); return { feedback: `고칠 곳: ${s.notes.join(", ")}` }; }, { input: Private })
  .addEdge(START, "analyze").addEdge("analyze", "write").addEdge("write", END)
  .compile();
await showGraph(privateDemo);
console.log("[끝]   결과", await privateDemo.invoke({ text: "He go to school every day." }));

**돌려 보면** analyze 는 `text` 를 받고, write 는 `notes` 만 받습니다. text 는 write 에게 가지 않습니다. 그리고 결과에는 `text` 와 `feedback` 만 있고 `notes` 는 없습니다. notes 가 그래프 State 에 없기 때문입니다.

이렇게 노드끼리만 주고받고 밖으로는 안 나가는 값을 **private state** 라고 부릅니다. analyze 는 Essay 에 없는 notes 를 돌려줬는데도 버려지지 않았습니다. write 의 입구 schema 에 notes 가 있어서, 그래프가 notes 필드를 알고 있기 때문입니다.

### 그래프의 입구와 출구 — input / output schema

이번에는 노드 하나가 아니라 그래프 전체입니다. 지금까지는 그래프의 입력과 출력이 State 전체였습니다. 그런데 학생은 글을 넣고 피드백만 받으면 됩니다. 분석기들이 주고받은 `notes` 는 그래프 안에서만 쓰는 메모입니다.

`StateGraph` 에 State 전체와 함께 **입구(input)** 와 **출구(output)** schema 를 따로 줄 수 있습니다.

In [ ]:
// L3 · 입구와 출구: 밖에서는 글을 넣고 피드백만 받는다. notes 는 그래프 안에서만 쓴다
const Overall = Annotation.Root({
  text: Annotation<string>(),
  notes: Annotation<string[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
  feedback: Annotation<string>(),
});
const Input = Annotation.Root({ text: Annotation<string>() });
const Output = Annotation.Root({ feedback: Annotation<string>() });
const tutor = new StateGraph(Overall, { input: Input, output: Output })
  .addNode("grammar", () => ({ notes: ["3인칭 단수 -s"] }))
  .addNode("vocab", () => ({ notes: ["ubiquitous → everywhere"] }))
  .addNode("write", (s) => ({ feedback: `고칠 곳 ${s.notes.length}개: ${s.notes.join(", ")}` }))
  .addEdge(START, "grammar").addEdge(START, "vocab")
  .addEdge(["grammar", "vocab"], "write")              // 둘 다 끝나면 write
  .addEdge("write", END)
  .compile();
await showGraph(tutor);
console.log(await tutor.invoke({ text: "Smartphone are ubiquitous." }));

**돌려 보면** `{ feedback: "고칠 곳 2개: …" }` 하나만 나옵니다. `notes` 는 안에서 잘 쓰였지만 출구에 없으니 밖으로 나오지 않습니다. `addEdge(["grammar", "vocab"], "write")` 는 두 노드가 다 끝나면 write 로 가라는 뜻입니다.

마지막으로 함정 하나입니다. 노드가 State 에 **없는** 필드를 돌려주면 어떻게 될까요? 필드 이름에 오타를 내 보겠습니다.

In [ ]:
// 함정: State 에 없는 필드는 에러 없이 버려진다
const Small = Annotation.Root({ feedback: Annotation<string>() });
const typo = new StateGraph(Small)
  .addNode("write", () => ({ feedbak: "오타 난 필드 이름" } as any))
  .addEdge(START, "write").addEdge("write", END)
  .compile();
await showGraph(typo);
console.log(await typo.invoke({ feedback: "처음 값" }));

**돌려 보면** `{ feedback: "처음 값" }` 입니다. `feedbak` 은 에러 없이 사라졌습니다. 노드가 분명히 돌려줬는데 State 에 안 보이면, 먼저 필드 이름부터 보세요.